In [ ]:
import os
import time

from pyspark.sql import SparkSession

# --- 1. Настройки окружения для macOS ---
# На macOS иногда возникают проблемы с форком процессов Java (Executor'ы не стартуют).
# Эта переменная часто решает проблему "Connection refused" или краши при запуске local-cluster
import platform
if platform.system() == 'Darwin':
    os.environ['OBJC_DISABLE_INITIALIZE_FORK_SAFETY'] = 'YES'
    
# Очистка старых сессий и переменных
try:
    existing_spark = SparkSession.getActiveSession()
    if existing_spark:
        existing_spark.stop()
        print("✅ Существующая сессия остановлена.")
except:
    pass

for key in list(os.environ.keys()):
    if 'SPARK' in key or 'JAVA_OPTS' in key:
        del os.environ[key]

# --- 2. Конфигурация Кластера ---
# Формат: local-cluster[число_воркеров, ядер_на_воркер, память_на_воркер_в_МБ]
# Мы просим 2 экзекутора, по 1 ядру, по 2 ГБ памяти каждый
NUM_EXECUTORS = 2
CORES_PER_EXECUTOR = 7
MEMORY_PER_EXECUTOR_MB = 8192 

MASTER_URL = f"local-cluster[{NUM_EXECUTORS}, {CORES_PER_EXECUTOR}, {MEMORY_PER_EXECUTOR_MB}]"

print(f"🚀 Запуск в режиме: {MASTER_URL}")

sp_s = (SparkSession.builder
    .master(MASTER_URL)
    .appName("LocalClusterTest")
    # Память драйвера (остается у вас)
    .config("spark.driver.memory", "6g") 
    # Память экзекутора (должна соответствовать или быть меньше чем в master URL)
    .config("spark.executor.memory", "8g")
    .config("spark.executor.cores", "6")
    .config("spark.executor.instances", NUM_EXECUTORS)
    # Увеличиваем память под оверхед, чтобы избежать ошибок выделения памяти
    .config("spark.memory.fraction", "0.6")
    .config("spark.sql.shuffle.partitions", "200")
    .getOrCreate()
)

sp_s.sparkContext.setLogLevel("ERROR")

# --- 3. Проверка конфигурации ---
print(f"✅ Сессия создана.")
print(f"Driver Memory Config: {sp_s.conf.get('spark.driver.memory')}")
print(f"Executor Memory Config: {sp_s.conf.get('spark.executor.memory')}")

# Проверка количества экзекуторов (может занять пару секунд на старт)
time.sleep(3) 
num_executors = len(sp_s.sparkContext.parallelize(range(10), NUM_EXECUTORS).glom().collect())
print(f"📊 Активных экзекуторов (проверка через RDD): {num_executors}")

# --- 4. Тест на распределение ---
# Чтобы убедиться, что задача ушла на экзекуторы, а не осталась на драйвере
def print_executor_info(iterator):
    # Получаем ID экзекутора из переменных окружения процесса
    executor_id = os.environ.get('SPARK_EXECUTOR_ID', 'Driver/Local')
    process_id = os.getpid()
    return [f"Executor ID: {executor_id}, PID: {process_id}"]

# Создаем датафрейм и применяем трансформацию
df = sp_s.range(0, 10, 1, 4) # 4 партиции
result = df.rdd.mapPartitions(print_executor_info).collect()

print("\n🖥️ Где выполнялись задачи:")
for line in result:
    print(line)

# Не забывайте останавливать сессию в конце скрипта, так как процессы тяжелые
# sp_s.stop() 

🚀 Запуск в режиме: local-cluster[2, 7, 8192]


26/04/07 19:03:49 WARN Utils: Your hostname, MacBook-Pro-Danil.local resolves to a loopback address: 127.0.0.1; using 10.246.62.43 instead (on interface en0)
26/04/07 19:03:49 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/07 19:03:50 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


✅ Сессия создана.
Driver Memory Config: 6g
Executor Memory Config: 8g


📊 Активных экзекуторов (проверка через RDD): 2

🖥️ Где выполнялись задачи:
Executor ID: Driver/Local, PID: 35027
Executor ID: Driver/Local, PID: 35026
Executor ID: Driver/Local, PID: 35032
Executor ID: Driver/Local, PID: 35033


----------------------------------------
Exception occurred during processing of request from ('127.0.0.1', 57351)
Traceback (most recent call last):
  File "/opt/homebrew/Cellar/python@3.11/3.11.14_3/Frameworks/Python.framework/Versions/3.11/lib/python3.11/socketserver.py", line 317, in _handle_request_noblock
    self.process_request(request, client_address)
  File "/opt/homebrew/Cellar/python@3.11/3.11.14_3/Frameworks/Python.framework/Versions/3.11/lib/python3.11/socketserver.py", line 348, in process_request
    self.finish_request(request, client_address)
  File "/opt/homebrew/Cellar/python@3.11/3.11.14_3/Frameworks/Python.framework/Versions/3.11/lib/python3.11/socketserver.py", line 361, in finish_request
    self.RequestHandlerClass(request, client_address, self)
  File "/opt/homebrew/Cellar/python@3.11/3.11.14_3/Frameworks/Python.framework/Versions/3.11/lib/python3.11/socketserver.py", line 755, in __init__
    self.handle()
  File "/Users/danilsamsutdinov/HypEx/.venv/lib/pytho

In [2]:
sp_s

In [3]:
import numpy as np
import pandas as pd
import pyspark.pandas as ps
import warnings
from pyspark.pandas.utils import PandasAPIOnSparkAdviceWarning

warnings.filterwarnings("ignore", category=PandasAPIOnSparkAdviceWarning)
import random
from hypex import AATest
from hypex.analyzers.aa import AAScoreAnalyzer, OneAAStatAnalyzer
from hypex.comparators import StatsTTest, StatsChi2Test, GroupSizes
from hypex.comparators.stats_hypothesis_testing import StatsTTest
from hypex.dataset import ExperimentData, AdditionalTreatmentRole, TargetRole
from hypex.dataset import (
    Dataset, SmallDataset, InfoRole, TargetRole, 
    TreatmentRole, DatasetAdapter, DefaultRole, StratificationRole
)
from hypex.dataset.roles import TargetRole, FeatureRole, InfoRole, StatisticRole
from hypex.experiments.base import Experiment, OnRoleExperiment
from hypex.experiments.base_complex import ParamsExperiment
from hypex.reporters import DatasetReporter
from hypex.reporters.aa import OneAADictReporter
from hypex.splitters import AASplitter
from hypex.splitters.aa import AASplitter
from hypex.utils import BackendsEnum
from hypex.ui.aa import AAOutput
from hypex.ui.base import ExperimentShell

/Users/danilsamsutdinov/HypEx/.venv/lib/python3.11/site-packages/pyspark/pandas/__init__.py:50: UserWarning: 'PYARROW_IGNORE_TIMEZONE' environment variable was not set. It is required to set this environment variable to '1' in both driver and executor sides if you use pyarrow>=2.0.0. pandas-on-Spark will set it for you but it does not work if there is a Spark context already launched.
  warnings.warn(


In [4]:
def generate_test_data(n_rows=100, filename=None, return_df=True):
    # Используем современный генератор случайных чисел (быстрее и надёжнее)
    rng = np.random.default_rng()
    
    # Пропускаем каждый 10-й ID (кроме 0), поэтому оставляем ~90% кандидатов.
    # Добавляем небольшой запас, чтобы гарантированно получить n_rows строк.
    n_candidates = int(np.ceil(n_rows * 10 / 9)) + 10

    # 1. Векторизованная генерация всех массивов одним вызовом
    ids = np.arange(n_candidates)
    signup_month = rng.integers(0, 12, size=n_candidates).astype(float)
    treat = rng.integers(0, 2, size=n_candidates).astype(float)
    pre_spends = rng.uniform(450, 550, size=n_candidates)
    age = rng.integers(18, 71, size=n_candidates).astype(float)
    gender = rng.choice(['M', 'F'], size=n_candidates)
    industry = rng.choice(['Logistics', 'E-commerce'], size=n_candidates)

    # 2. Векторизованный расчёт post_spends
    mult_low = rng.uniform(0.8, 0.9, size=n_candidates)
    mult_high = rng.uniform(1.0, 1.1, size=n_candidates)
    post_spends = np.where(treat == 0, pre_spends * mult_low, pre_spends * mult_high)

    # 3. Округление (работает так же, как встроенный round)
    pre_spends = np.round(pre_spends, 1)
    post_spends = np.round(post_spends, 1)

    # 4. Фильтрация ID (пропускаем каждый 10-й, кроме 0) и обрезка до n_rows
    mask = (ids == 0) | (ids % 10 != 0)
    valid_idx = np.where(mask)[0][:n_rows]

    # 5. Сборка DataFrame
    df = pd.DataFrame({
        'user_id': ids[valid_idx],
        'signup_month': signup_month[valid_idx],
        'treat': treat[valid_idx],
        'pre_spends': pre_spends[valid_idx],
        'post_spends': post_spends[valid_idx],
        'age': age[valid_idx],
        'gender': gender[valid_idx],
        'industry': industry[valid_idx]
    })

    if filename:
        df.to_csv(filename, index=False, encoding='utf-8')
        print(f"Сгенерировано {n_rows} строк в файле {filename}")
    
    if return_df:
        return df

In [5]:
def create_spark_ds(n_rows: int = 100):
    return Dataset(
        roles={
            "user_id": InfoRole(float),
            "treat": TreatmentRole(),
            "pre_spends": TargetRole(),
            "post_spends": DefaultRole(),
            "gender": StratificationRole(str),
            "industry" : DefaultRole()
        }, 
        data=generate_test_data(n_rows=n_rows), 
        session=sp_s,
        backend=BackendsEnum.spark
    )

In [6]:
def AATest_without_analyse(num_splits: int = 1) -> ParamsExperiment:
    base_experiment = Experiment(
        executors=[
            AASplitter(
                control_size=0.5, 
                constant_key=True,
                save_groups=True
            ),
            StatsTTest(
                grouping_role=AdditionalTreatmentRole(),
                target_roles=TargetRole(),
                reliability=0.05
            ),
            OneAAStatAnalyzer()
        ]
    )

    return ParamsExperiment(
        executors=[base_experiment],
        params={
            AASplitter: {
                "random_state": range(num_splits),
                "control_size": [0.5]
            }
        },
        reporter=DatasetReporter(OneAADictReporter(front=False))
    )

In [ ]:
ds = create_spark_ds(n_rows=2_000_000)
ds.persist()
exp_data = ExperimentData(ds)
result = AATest_without_analyse().execute(exp_data)

In [ ]:
# executor.key = AASplitter┴rs 0┴; dt = 10.1766c
# 100%|██████████| 1/1 [00:32<00:00, 33.00s/it]                                   
# executor.key = StatsTTest┴┴pre_spends; dt = 18.6264c
# executor.key = OneAAStatAnalyzer┴┴; dt = 0.0018c

In [ ]:
# executor.key = AASplitter┴rs 0┴; dt = 6.0550c
# 100%|██████████| 1/1 [00:21<00:00, 21.52s/it]                                   
# executor.key = StatsTTest┴┴pre_spends; dt = 13.1315c
# executor.key = OneAAStatAnalyzer┴┴; dt = 0.0022c

In [ ]:
# executor.key = AASplitter┴rs 0┴; dt = 8.5361c
# 100%|██████████| 1/1 [00:34<00:00, 34.08s/it]                                   
# executor.key = StatsTTest┴┴pre_spends; dt = 23.4454c
# executor.key = OneAAStatAnalyzer┴┴; dt = 0.0024c

In [ ]:
import cProfile
import pstats

pr = cProfile.Profile()
pr.enable()

exp_data = ExperimentData(create_spark_ds(n_rows=1_000_000))
result = AATest_without_analyse().execute(exp_data)

pr.disable()
stats = pstats.Stats(pr).sort_stats('cumulative')
stats.print_stats(100)  # топ-20 функций по суммарному времени

In [ ]:
exp_data = ExperimentData(create_spark_ds(n_rows=100000))
result = AATest_without_analyse().execute(exp_data)

In [ ]:
# exp_data.analysis_tables

In [7]:
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import psutil
import os
import gc
import requests
import threading
from datetime import datetime

# ================= КОНФИГУРАЦИЯ =================
class BenchmarkConfig:
    N_ROWS_VALUES = [10000, 100000, 250000, 500000, 750000, 1000000, 1500000, 2000000, 2500000, 3000000, 4000000]
    # N_ROWS_VALUES = [20_000_000]
    N_REPEATS = 10
    MEMORY_SAMPLE_INTERVAL = 0.01  # Уменьшили интервал для точности
    SAVE_PATH = 'spark_benchmark_results'
    SPARK_UI_URL = "http://localhost:4040"
    DEBUG_API = False  # Включите True, чтобы видеть сырые ответы API
    
    @classmethod
    def ensure_dir(cls):
        os.makedirs(cls.SAVE_PATH, exist_ok=True)

# ================= МОНТОРИНГ ПАМЯТИ (FIXED) =================
class ClusterMemoryMonitor:
    def __init__(self, spark_session):
        self.spark = spark_session
        self.jvm = spark_session._jvm
        self.process = psutil.Process(os.getpid())
        self.samples = []
        self.is_monitoring = False
        self.app_id = None
        self._api_failures = 0
        
    def _get_executor_metrics(self):
        """Запрашивает метрики экзекуторов через Spark UI API"""
        if not self.app_id:
            try:
                self.app_id = self.spark.sparkContext.applicationId
            except:
                return {'total_mem_mb': 0, 'count': 0}
        
        try:
            url = f"{BenchmarkConfig.SPARK_UI_URL}/api/v1/applications/{self.app_id}/executors"
            response = requests.get(url, timeout=2)
            
            if response.status_code != 200:
                return {'total_mem_mb': 0, 'count': 0}
                
            executors = response.json()
            workers = [e for e in executors if e.get('id') != 'driver']
            
            total_peak_heap = 0      # JVM Heap пик
            total_peak_exec = 0      # Execution memory пик
            total_storage = 0        # Текущая storage memory (кэш)
            
            for e in workers:
                # 1. Peak memory (остаётся после завершения задач) - ГЛАВНОЕ
                peak = e.get('peakMemoryMetrics', {})
                if peak:
                    # Пиковая JVM heap (в байтах)
                    jvm_heap_peak = peak.get('JVMHeapMemory', 0) or 0
                    # Пиковая память под вычисления
                    exec_mem_peak = peak.get('OnHeapExecutionMemory', 0) or 0
                    total_peak_heap += jvm_heap_peak
                    total_peak_exec += exec_mem_peak
                
                # 2. Текущая storage memory (для полноты картины)
                mem_metrics = e.get('memoryMetrics', {})
                if mem_metrics:
                    used_storage = (
                        (mem_metrics.get('usedOnHeapStorageMemory', 0) or 0) +
                        (mem_metrics.get('usedOffHeapStorageMemory', 0) or 0)
                    )
                    total_storage += used_storage
            
            if BenchmarkConfig.DEBUG_API and workers:
                print(f"🔍 API: {len(workers)} executors | "
                    f"Peak Heap: {total_peak_heap/1024/1024:.1f} MB | "
                    f"Peak Exec: {total_peak_exec/1024/1024:.1f} MB")
            
            return {
                'total_peak_heap_mb': total_peak_heap / 1024 / 1024,      # ← Главное!
                'total_peak_exec_mb': total_peak_exec / 1024 / 1024,      # ← Для анализа вычислений
                'total_storage_mb': total_storage / 1024 / 1024,
                'count': len(workers)
            }
            
        except Exception as e:
            if BenchmarkConfig.DEBUG_API:
                print(f"⚠️ API Error: {e}")
            return {'total_peak_heap_mb': 0, 'total_peak_exec_mb': 0, 'total_storage_mb': 0, 'count': 0}

    def start(self):
        self.samples = []
        self.is_monitoring = True
        self._api_failures = 0
        if not self.app_id:
            try: 
                self.app_id = self.spark.sparkContext.applicationId
            except: 
                pass
        
    def stop(self):
        self.is_monitoring = False
        
    def sample(self):
        """Снимает текущий срез метрик памяти"""
        if not self.is_monitoring:
            return
            
        try:
            runtime = self.jvm.Runtime.getRuntime()
            exec_metrics = self._get_executor_metrics()  # Возвращает новые ключи
            
            sample = {
                'timestamp': time.time(),
                
                # --- DRIVER METRICS ---
                'driver_heap_used_mb': (runtime.totalMemory() - runtime.freeMemory()) / 1024 / 1024,
                'driver_heap_max_mb': runtime.maxMemory() / 1024 / 1024,
                'driver_rss_mb': self.process.memory_info().rss / 1024 / 1024,
                'driver_cpu_percent': self.process.cpu_percent(interval=0.05),
                
                # --- EXECUTOR METRICS (ИСПРАВЛЕНО: новые ключи) ---
                'executor_peak_heap_mb': exec_metrics.get('total_peak_heap_mb', 0),
                'executor_peak_exec_mb': exec_metrics.get('total_peak_exec_mb', 0),
                'executor_storage_mb': exec_metrics.get('total_storage_mb', 0),
                'executor_count': exec_metrics.get('count', 0),
            }
            
            self.samples.append(sample)
            
        except Exception as e:
            print(f"⚠️ Ошибка сбора метрик: {e}")
            # Для отладки: раскомментируйте, чтобы увидеть, что приходит от API
            # import traceback; traceback.print_exc()
    
    def get_statistics(self):
        """Агрегирует статистику по собранным сэмплам"""
        if not self.samples:
            return {}
            
        df = pd.DataFrame(self.samples)
        
        stats = {
            # --- DRIVER ---
            'driver_heap_avg_mb': df['driver_heap_used_mb'].mean(),
            'driver_heap_max_mb': df['driver_heap_used_mb'].max(),
            'driver_rss_avg_mb': df['driver_rss_mb'].mean(),
            'driver_rss_max_mb': df['driver_rss_mb'].max(),
            'driver_cpu_avg': df['driver_cpu_percent'].mean(),
            
            # --- EXECUTORS (ОБНОВЛЕНО) ---
            'executor_peak_heap_avg_mb': df['executor_peak_heap_mb'].mean(),
            'executor_peak_heap_max_mb': df['executor_peak_heap_mb'].max(),
            'executor_peak_exec_avg_mb': df['executor_peak_exec_mb'].mean(),
            'executor_peak_exec_max_mb': df['executor_peak_exec_mb'].max(),
            'executor_storage_avg_mb': df['executor_storage_mb'].mean(),
            'executor_storage_max_mb': df['executor_storage_mb'].max(),
            
            'executor_count': df['executor_count'].max(),
            'samples_count': len(df),
        }
        return stats

# ================= ФОНОВЫЙ СБОРЩИК (FIXED TIMING) =================
class BackgroundMonitor:
    def __init__(self, memory_monitor, interval=0.2):
        self.monitor = memory_monitor
        self.interval = interval
        self.running = False
        
    def start(self):
        self.running = True
        self.monitor.start()
        
    def stop(self):
        self.running = False
        self.monitor.stop()
        
    def run_and_collect(self, func, *args, **kwargs):
        import threading
        
        def collect_loop():
            while self.running:
                self.monitor.sample()
                time.sleep(self.interval)
        
        collector = threading.Thread(target=collect_loop, daemon=True)
        self.start()
        collector.start()
        
        # Выполнение задачи
        result = func(*args, **kwargs)
        
        # === КРИТИЧЕСКОЕ ИСПРАВЛЕНИЕ ===
        # Ждем 2 секунды после завершения задачи, чтобы успеть снять пики памяти экзекуторов
        # before они освободятся
        time.sleep(2.0)
        
        # Делаем еще несколько сэмплов перед остановкой
        for _ in range(3):
            self.monitor.sample()
            time.sleep(0.1)
        # ===============================
        
        self.stop()
        collector.join(timeout=1)
        
        return result, self.monitor.get_statistics()

# ================= БЕНЧМАРК =================
def run_single_test(spark_ds, spark_session, monitor):
    exp_data = ExperimentData(spark_ds)
    AATest_without_analyse().execute(exp_data)

def run_benchmark_with_memory(spark_session):
    BenchmarkConfig.ensure_dir()
    monitor = ClusterMemoryMonitor(spark_session)
    bg_monitor = BackgroundMonitor(monitor, interval=BenchmarkConfig.MEMORY_SAMPLE_INTERVAL)
    
    results = []
    print(f"🚀 Старт бенчмарка: {datetime.now().strftime('%H:%M:%S')}")
    print(f"💾 Driver Max Heap: {spark_session._jvm.Runtime.getRuntime().maxMemory() / 1024 / 1024 / 1024:.2f} GB")
    print(f"🌐 Spark UI: {BenchmarkConfig.SPARK_UI_URL}")
    print("-" * 70)
    
    for n_rows in BenchmarkConfig.N_ROWS_VALUES:
        spark_ds = create_spark_ds(n_rows=n_rows)
        spark_ds.persist()
        print(f"\n📊 n_rows = {n_rows:,}")
        
        gc.collect()
        time.sleep(0.5)
        
        times = []
        memory_stats_list = []
        
        for i in range(BenchmarkConfig.N_REPEATS):
            try:
                start = time.perf_counter()
                
                _, mem_stats = bg_monitor.run_and_collect(
                    run_single_test, spark_ds, spark_session, monitor
                )
                
                end = time.perf_counter()
                duration = end - start
                times.append(duration)
                memory_stats_list.append(mem_stats)
                
                print(f"   Повтор {i+1}: {duration:.3f} сек | "
                      f"Driver RSS: {mem_stats.get('driver_rss_max_mb', 0):.0f} MB | "
                      f"Exec Mem: {mem_stats.get('executor_peak_heap_max_mb', 0):.0f} MB | "
                      f"Exec Samples: {mem_stats.get('executor_samples_nonzero', 0)}")
                
                if mem_stats.get('api_failures', 0) > 0:
                    print(f"   ⚠️ API Failures: {mem_stats.get('api_failures')}")
                
            except Exception as e:
                print(f"   ❌ Ошибка: {e}")
                times.append(None)
                memory_stats_list.append({})
        
        valid_times = [t for t in times if t is not None]
        if valid_times:
            avg_mem_stats = {
                k: np.mean([s.get(k, 0) for s in memory_stats_list]) 
                for k in memory_stats_list[0].keys() if k not in ['samples_count', 'executor_samples_nonzero', 'api_failures']
            }
            max_mem_stats = {
                k: np.max([s.get(k, 0) for s in memory_stats_list]) 
                for k in memory_stats_list[0].keys() if k not in ['samples_count', 'executor_samples_nonzero', 'api_failures']
            }
            
            results.append({
                'n_rows': n_rows,
                'time_mean_sec': np.mean(valid_times),
                'time_std_sec': np.std(valid_times),
                
                # Driver
                'driver_rss_max_mb': max_mem_stats.get('driver_rss_max_mb', 0),
                'driver_heap_max_mb': max_mem_stats.get('driver_heap_max_mb', 0),
                
                # Executors (ОБНОВЛЕНО)
                'executor_peak_heap_max_mb': max_mem_stats.get('executor_peak_heap_max_mb', 0),
                'executor_peak_exec_max_mb': max_mem_stats.get('executor_peak_exec_max_mb', 0),
                'executor_storage_max_mb': max_mem_stats.get('executor_storage_max_mb', 0),
                
                'driver_cpu_avg': avg_mem_stats.get('driver_cpu_avg', 0),
                'rows_per_sec': n_rows / np.mean(valid_times),
            })
        else:
            results.append({'n_rows': n_rows, 'time_mean_sec': None})
        spark_ds.unpersist()
    
    print(f"\n🏁 Завершено: {datetime.now().strftime('%H:%M:%S')}")
    return pd.DataFrame(results)

In [8]:
def plot_comprehensive_results(df):
    """Строит расширенные графики для Driver vs Executors"""
    df_clean = df.dropna(subset=['time_mean_sec']).copy()
    if len(df_clean) == 0:
        print("⚠️ Нет данных для графиков")
        return

    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle('📈 Cluster Benchmark: Driver vs Executors Memory', fontsize=14, fontweight='bold')
    
    # === 1. Время выполнения ===
    ax1 = axes[0, 0]
    ax1.errorbar(df_clean['n_rows'], df_clean['time_mean_sec'], 
                 yerr=df_clean['time_std_sec'], marker='o', capsize=5, color='black')
    ax1.set_xscale('log')
    ax1.set_xlabel('n_rows')
    ax1.set_ylabel('Время (сек)')
    ax1.set_title('⏱️ Время выполнения')
    ax1.grid(True, alpha=0.3)
    
    # === 2. Driver Memory (RSS + Heap) ===
    ax2 = axes[0, 1]
    ax2.plot(df_clean['n_rows'], df_clean['driver_rss_max_mb'], 
             marker='s', label='Driver RSS (System)', color='blue', linewidth=2)
    ax2.plot(df_clean['n_rows'], df_clean['driver_heap_max_mb'], 
             marker='^', label='Driver Heap (JVM)', color='cyan', linestyle='--')
    ax2.set_xscale('log')
    ax2.set_xlabel('n_rows')
    ax2.set_ylabel('Память (MB)')
    ax2.set_title('🖥️ Память Драйвера')
    ax2.legend(fontsize=8)
    ax2.grid(True, alpha=0.3)
    
    # === 3. Executor Memory (Peak) ===
    ax3 = axes[0, 2]
    # Stack plot: Execution + Storage
    if 'executor_peak_exec_max_mb' in df_clean.columns:
        ax3.fill_between(df_clean['n_rows'], 
                        df_clean['executor_storage_max_mb'], 
                        df_clean['executor_peak_exec_max_mb'] + df_clean['executor_storage_max_mb'],
                        label='Execution Memory', color='orange', alpha=0.6)
    ax3.plot(df_clean['n_rows'], df_clean['executor_storage_max_mb'], 
             marker='d', label='Storage (Cache)', color='red')
    ax3.plot(df_clean['n_rows'], df_clean['executor_peak_heap_max_mb'], 
             marker='*', label='Peak JVM Heap (Total)', color='darkred', linewidth=2)
    ax3.set_xscale('log')
    ax3.set_xlabel('n_rows')
    ax3.set_ylabel('Память (MB)')
    ax3.set_title('⚙️ Память Экзекуторов (Пик)')
    ax3.legend(fontsize=8)
    ax3.grid(True, alpha=0.3)
    
    # === 4. Производительность ===
    ax4 = axes[1, 0]
    ax4.plot(df_clean['n_rows'], df_clean['rows_per_sec'], marker='o', color='green')
    ax4.set_xscale('log')
    ax4.set_xlabel('n_rows')
    ax4.set_ylabel('строк/сек')
    ax4.set_title('🚀 Пропускная способность')
    ax4.grid(True, alpha=0.3)
    
    # === 5. Сравнение: Driver RSS vs Executors Peak Heap ===
    ax5 = axes[1, 1]
    x = np.arange(len(df_clean))
    width = 0.35
    
    bars1 = ax5.bar(x - width/2, df_clean['driver_rss_max_mb'], width, 
                   label='Driver RSS', color='blue', alpha=0.7)
    bars2 = ax5.bar(x + width/2, df_clean['executor_peak_heap_max_mb'], width, 
                   label='Executors Peak Heap', color='red', alpha=0.7)
    
    ax5.set_xticks(x)
    ax5.set_xticklabels([f"{n/1e6:.1f}M" for n in df_clean['n_rows']], rotation=45, ha='right')
    ax5.set_ylabel('Память (MB)')
    ax5.set_title('⚖️ Пик памяти: Драйвер vs Экзекуторы')
    ax5.legend()
    ax5.grid(True, alpha=0.3, axis='y')
    
    # Добавляем аннотации с коэффициентами
    for i, (d_mem, e_mem) in enumerate(zip(df_clean['driver_rss_max_mb'], df_clean['executor_peak_heap_max_mb'])):
        if d_mem > 0:
            ratio = e_mem / d_mem
            ax5.text(i, max(d_mem, e_mem) * 1.05, f'{ratio:.1f}×', 
                    ha='center', fontsize=9, fontweight='bold')
    
    # === 6. Memory Efficiency: MB per 1M rows ===
    ax6 = axes[1, 2]
    
    # Считаем эффективность для драйвера и экзекуторов
    driver_eff = df_clean['driver_rss_max_mb'] / df_clean['n_rows'] * 1_000_000
    exec_eff = df_clean['executor_peak_heap_max_mb'] / df_clean['n_rows'] * 1_000_000
    
    ax6.plot(df_clean['n_rows'], driver_eff, marker='s', label='Driver MB/1M rows', color='blue')
    ax6.plot(df_clean['n_rows'], exec_eff, marker='d', label='Executors MB/1M rows', color='red')
    ax6.axhline(y=driver_eff.mean(), color='blue', linestyle=':', alpha=0.3)
    ax6.axhline(y=exec_eff.mean(), color='red', linestyle=':', alpha=0.3)
    
    ax6.set_xscale('log')
    ax6.set_xlabel('n_rows')
    ax6.set_ylabel('MB на 1 млн строк')
    ax6.set_title('📐 Эффективность памяти')
    ax6.legend(fontsize=8)
    ax6.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(f'{BenchmarkConfig.SAVE_PATH}/cluster_analysis.png', dpi=300, bbox_inches='tight')
    print(f"📁 Графики сохранены в {BenchmarkConfig.SAVE_PATH}/cluster_analysis.png")
    plt.show()

In [9]:
def print_memory_analysis(df):
    """Выводит текстовый анализ памяти для Driver и Executors"""
    df_clean = df.dropna(subset=['time_mean_sec']).copy()
    if len(df_clean) < 1:
        print("⚠️ Недостаточно данных для анализа памяти")
        return

    print("\n" + "="*70)
    print("🔍 АНАЛИЗ ПАМЯТИ: DRIVER vs EXECUTORS")
    print("="*70)
    
    last = df_clean.iloc[-1]
    first = df_clean.iloc[0]
    
    # === 1. Пиковые значения ===
    print(f"\n📊 Пиковые значения (на {last['n_rows']:,} строк):")
    print(f"   ┌─ Driver:")
    print(f"   │  • RSS Max:     {last['driver_rss_max_mb']:>8.1f} MB  ({last['driver_rss_max_mb']/1024:.2f} GB)")
    print(f"   │  • Heap Max:    {last['driver_heap_max_mb']:>8.1f} MB  ({last['driver_heap_max_mb']/1024:.2f} GB)")
    print(f"   └─ Executors (сумма):")
    print(f"      • Peak Heap:   {last['executor_peak_heap_max_mb']:>8.1f} MB  ({last['executor_peak_heap_max_mb']/1024:.2f} GB)")
    print(f"      • Exec Memory: {last['executor_peak_exec_max_mb']:>8.1f} MB  (под вычисления)")
    print(f"      • Storage:     {last['executor_storage_max_mb']:>8.1f} MB  (кэш RDD)")
    
    # === 2. Масштабирование ===
    if first['n_rows'] > 0 and last['n_rows'] > first['n_rows']:
        data_growth = last['n_rows'] / first['n_rows']
        driver_growth = last['driver_rss_max_mb'] / (first['driver_rss_max_mb'] + 1)
        exec_growth = last['executor_peak_heap_max_mb'] / (first['executor_peak_heap_max_mb'] + 1)
        
        print(f"\n📏 Масштабирование (×{data_growth:.1f} данных):")
        print(f"   • Driver RSS:     ×{driver_growth:.2f}  {'⚠️ Нелинейно!' if driver_growth > data_growth * 1.3 else '✓'}")
        print(f"   • Executors Heap: ×{exec_growth:.2f}  {'⚠️ Нелинейно!' if exec_growth > data_growth * 1.3 else '✓'}")
    
    # === 3. Баланс нагрузки ===
    print(f"\n⚖️ Баланс памяти:")
    driver_mem = last['driver_rss_max_mb']
    exec_mem = last['executor_peak_heap_max_mb']
    total_mem = driver_mem + exec_mem
    
    if total_mem > 0:
        driver_pct = driver_mem / total_mem * 100
        exec_pct = exec_mem / total_mem * 100
        
        print(f"   • Driver:     {driver_pct:5.1f}%  {'████' * int(driver_pct/25)}")
        print(f"   • Executors:  {exec_pct:5.1f}%  {'████' * int(exec_pct/25)}")
        
        # Анализ дисбаланса
        ratio = exec_mem / (driver_mem + 1)  # +1 to avoid div by zero
        
        if ratio > 10:
            print(f"\n   ⚠️  Экзекуторы потребляют в {ratio:.1f}× больше драйвера")
            print(f"      → Это нормально для тяжелых трансформаций (groupBy, join)")
        elif ratio < 0.3 and driver_mem > 500:
            print(f"\n   ⚠️  Драйвер потребляет непропорционально много ({driver_pct:.1f}%)")
            print(f"      → Возможные причины:")
            print(f"         • .collect() собирает данные на драйвер")
            print(f"         • Недостаточно партиций (данные не распараллеливаются)")
            print(f"         • Broadcast-join тянет таблицу в драйвер")
        else:
            print(f"\n   ✓ Баланс в норме (коэффициент: {ratio:.2f})")
    
    # === 4. Эффективность ===
    print(f"\n⚡ Эффективность использования памяти:")
    
    # MB per 1M rows
    driver_eff = last['driver_rss_max_mb'] / last['n_rows'] * 1_000_000
    exec_eff = last['executor_peak_heap_max_mb'] / last['n_rows'] * 1_000_000
    
    print(f"   • Driver:     {driver_eff:6.2f} MB / 1M rows")
    print(f"   • Executors:  {exec_eff:6.2f} MB / 1M rows")
    
    if exec_eff > 0 and driver_eff / exec_eff > 3:
        print(f"   ⚠️  Драйвер неэффективен: тратит в {driver_eff/exec_eff:.1f}× больше на тот же объём данных")
    
    # === 5. Рекомендации ===
    print(f"\n💡 Рекомендации:")
    
    # Проверка лимитов
    driver_limit = last['driver_heap_max_mb']  # из конфига, примерно
    exec_limit_per = 2048  # например, 2 ГБ на экзекутор из conf
    n_executors = df_clean['executor_peak_heap_max_mb'].count()  # грубая оценка
    
    if last['executor_peak_heap_max_mb'] > exec_limit_per * 1.8:  # 90% от лимита
        print(f"   • ⚠️ Экзекуторы близки к лимиту памяти!")
        print(f"     → Увеличьте `spark.executor.memory` или оптимизируйте партиционирование")
    
    if last['driver_rss_max_mb'] > driver_limit * 0.9:
        print(f"   • ⚠️ Драйвер близок к лимиту heap!")
        print(f"     → Увеличьте `spark.driver.memory` или избегайте .collect() на больших данных")
    
    if 'executor_peak_exec_max_mb' in last and last['executor_peak_exec_max_mb'] > last['executor_storage_max_mb'] * 5:
        print(f"   • ℹ️  Большая часть памяти экзекуторов уходит на вычисления (не кэш)")
        print(f"     → Проверьте план выполнения: возможно, есть дорогие shuffle-операции")
    
    # Если экзекуторы почти не используют память
    if last['executor_peak_heap_max_mb'] < 100 and last['n_rows'] > 10000:
        print(f"   • ⚠️ Экзекуторы почти не используют память при больших данных")
        print(f"     → Проверьте, распределяются ли задачи (партиционирование, .repartition())")
    
    print("="*70)

In [10]:

# ================= ЗАПУСК =================
# Предполагается, что sp_s уже создана в режиме local-cluster
results_df = run_benchmark_with_memory(sp_s)
# plot_comprehensive_results(results_df)
# print_memory_analysis(results_df)

🚀 Старт бенчмарка: 19:04:08
💾 Driver Max Heap: 6.00 GB
🌐 Spark UI: http://localhost:4040
----------------------------------------------------------------------

📊 n_rows = 10,000


  0%|          | 0/1 [00:00<?, ?it/s]

executor.key = AASplitter┴rs 0┴; dt = 2.6068c


100%|██████████| 1/1 [00:05<00:00,  5.43s/it]

executor.key = StatsTTest┴┴pre_spends; dt = 2.6603c
executor.key = OneAAStatAnalyzer┴┴; dt = 0.0020c


...

100%|██████████| 1/1 [00:29<00:00, 29.67s/it]                                   


executor.key = StatsTTest┴┴pre_spends; dt = 21.4306c
executor.key = OneAAStatAnalyzer┴┴; dt = 0.0018c
   Повтор 9: 36.564 сек | Driver RSS: 893 MB | Exec Mem: 13219 MB | Exec Samples: 0


executor.key = AASplitter┴rs 0┴; dt = 5.8393c


100%|██████████| 1/1 [00:34<00:00, 34.01s/it]                                   


executor.key = StatsTTest┴┴pre_spends; dt = 24.1326c
executor.key = OneAAStatAnalyzer┴┴; dt = 0.0058c
   Повтор 10: 40.838 сек | Driver RSS: 764 MB | Exec Mem: 13219 MB | Exec Samples: 0

🏁 Завершено: 19:39:23


In [ ]:
# 🚀 Старт бенчмарка: 15:42:35
# 💾 Driver Max Heap: 4.00 GB
# 🌐 Spark UI: http://localhost:4040
# ----------------------------------------------------------------------

# 📊 n_rows = 10,000
# 100%|██████████| 1/1 [00:08<00:00,  8.39s/it]
#    Повтор 1: 11.190 сек | Driver RSS: 285 MB | Exec Mem: 1087 MB | Exec Samples: 0
# 100%|██████████| 1/1 [00:08<00:00,  8.25s/it]                                   
#    Повтор 2: 10.947 сек | Driver RSS: 285 MB | Exec Mem: 1102 MB | Exec Samples: 0
# 100%|██████████| 1/1 [00:07<00:00,  7.98s/it]                                   
#    Повтор 3: 10.630 сек | Driver RSS: 285 MB | Exec Mem: 1102 MB | Exec Samples: 0
# 100%|██████████| 1/1 [00:07<00:00,  7.89s/it]
#    Повтор 4: 10.574 сек | Driver RSS: 285 MB | Exec Mem: 1108 MB | Exec Samples: 0
# 100%|██████████| 1/1 [00:07<00:00,  7.73s/it]                                   
#    Повтор 5: 10.392 сек | Driver RSS: 285 MB | Exec Mem: 1108 MB | Exec Samples: 0
# 100%|██████████| 1/1 [00:07<00:00,  7.87s/it]                                   
#    Повтор 6: 10.516 сек | Driver RSS: 285 MB | Exec Mem: 1188 MB | Exec Samples: 0
# 100%|██████████| 1/1 [00:07<00:00,  7.74s/it]
#    Повтор 7: 10.379 сек | Driver RSS: 285 MB | Exec Mem: 1346 MB | Exec Samples: 0
# 100%|██████████| 1/1 [00:07<00:00,  7.63s/it]
#    Повтор 8: 10.285 сек | Driver RSS: 285 MB | Exec Mem: 1346 MB | Exec Samples: 0
# 100%|██████████| 1/1 [00:07<00:00,  7.93s/it]                                   
#    Повтор 9: 10.585 сек | Driver RSS: 285 MB | Exec Mem: 1346 MB | Exec Samples: 0
# 100%|██████████| 1/1 [00:07<00:00,  7.78s/it]
#    Повтор 10: 10.444 сек | Driver RSS: 285 MB | Exec Mem: 1401 MB | Exec Samples: 0

# 📊 n_rows = 100,000
# 100%|██████████| 1/1 [00:09<00:00,  9.23s/it]                                   
#    Повтор 1: 12.337 сек | Driver RSS: 383 MB | Exec Mem: 1562 MB | Exec Samples: 0
# 100%|██████████| 1/1 [00:08<00:00,  8.77s/it]                                   
#    Повтор 2: 12.242 сек | Driver RSS: 383 MB | Exec Mem: 1562 MB | Exec Samples: 0
# 100%|██████████| 1/1 [00:09<00:00,  9.07s/it]                                   
#    Повтор 3: 11.862 сек | Driver RSS: 383 MB | Exec Mem: 1789 MB | Exec Samples: 0
# 100%|██████████| 1/1 [00:09<00:00,  9.22s/it]                                   
#    Повтор 4: 12.004 сек | Driver RSS: 383 MB | Exec Mem: 1789 MB | Exec Samples: 0
# 100%|██████████| 1/1 [00:09<00:00,  9.05s/it]
#    Повтор 5: 11.782 сек | Driver RSS: 372 MB | Exec Mem: 2015 MB | Exec Samples: 0
# 100%|██████████| 1/1 [00:09<00:00,  9.02s/it]
#    Повтор 6: 11.753 сек | Driver RSS: 372 MB | Exec Mem: 2107 MB | Exec Samples: 0
# 100%|██████████| 1/1 [00:08<00:00,  8.80s/it]                                   
#    Повтор 7: 11.588 сек | Driver RSS: 372 MB | Exec Mem: 2107 MB | Exec Samples: 0
# 100%|██████████| 1/1 [00:08<00:00,  8.83s/it]
#    Повтор 8: 11.584 сек | Driver RSS: 372 MB | Exec Mem: 2179 MB | Exec Samples: 0
# 100%|██████████| 1/1 [00:08<00:00,  8.60s/it]                                   
#    Повтор 9: 11.338 сек | Driver RSS: 372 MB | Exec Mem: 2179 MB | Exec Samples: 0
# 100%|██████████| 1/1 [00:08<00:00,  8.58s/it]                                   
#    Повтор 10: 11.303 сек | Driver RSS: 372 MB | Exec Mem: 2229 MB | Exec Samples: 0

# 📊 n_rows = 250,000
# 100%|██████████| 1/1 [00:10<00:00, 10.44s/it]                                   
#    Повтор 1: 14.084 сек | Driver RSS: 574 MB | Exec Mem: 2412 MB | Exec Samples: 0
# 100%|██████████| 1/1 [00:10<00:00, 10.37s/it]                                   
#    Повтор 2: 13.278 сек | Driver RSS: 575 MB | Exec Mem: 2412 MB | Exec Samples: 0
# 100%|██████████| 1/1 [00:10<00:00, 10.47s/it]                                   
#    Повтор 3: 13.335 сек | Driver RSS: 531 MB | Exec Mem: 2412 MB | Exec Samples: 0
# 100%|██████████| 1/1 [00:10<00:00, 10.62s/it]                                   
#    Повтор 4: 14.212 сек | Driver RSS: 498 MB | Exec Mem: 2412 MB | Exec Samples: 0
# 100%|██████████| 1/1 [00:10<00:00, 10.54s/it]                                   
#    Повтор 5: 13.443 сек | Driver RSS: 498 MB | Exec Mem: 2517 MB | Exec Samples: 0
# 100%|██████████| 1/1 [00:10<00:00, 10.25s/it]                                   
#    Повтор 6: 13.167 сек | Driver RSS: 498 MB | Exec Mem: 2517 MB | Exec Samples: 0
# 100%|██████████| 1/1 [00:10<00:00, 10.34s/it]                                   
#    Повтор 7: 13.221 сек | Driver RSS: 498 MB | Exec Mem: 2517 MB | Exec Samples: 0
# 100%|██████████| 1/1 [00:10<00:00, 10.39s/it]                                   
#    Повтор 8: 13.274 сек | Driver RSS: 498 MB | Exec Mem: 2704 MB | Exec Samples: 0
# 100%|██████████| 1/1 [00:10<00:00, 10.71s/it]                                   
#    Повтор 9: 13.603 сек | Driver RSS: 498 MB | Exec Mem: 2704 MB | Exec Samples: 0
# 100%|██████████| 1/1 [00:10<00:00, 10.53s/it]                                   
#    Повтор 10: 13.372 сек | Driver RSS: 498 MB | Exec Mem: 2704 MB | Exec Samples: 0

# 📊 n_rows = 500,000
# 100%|██████████| 1/1 [00:13<00:00, 13.19s/it]                                   
#    Повтор 1: 17.639 сек | Driver RSS: 860 MB | Exec Mem: 2704 MB | Exec Samples: 0
# 100%|██████████| 1/1 [00:12<00:00, 12.56s/it]                                   
#    Повтор 2: 15.707 сек | Driver RSS: 860 MB | Exec Mem: 2710 MB | Exec Samples: 0
# 100%|██████████| 1/1 [00:12<00:00, 12.25s/it]                                   
#    Повтор 3: 15.430 сек | Driver RSS: 806 MB | Exec Mem: 2717 MB | Exec Samples: 0
# 100%|██████████| 1/1 [00:12<00:00, 12.36s/it]                                   
#    Повтор 4: 15.489 сек | Driver RSS: 699 MB | Exec Mem: 3092 MB | Exec Samples: 0
# 100%|██████████| 1/1 [00:12<00:00, 12.49s/it]                                   
#    Повтор 5: 15.585 сек | Driver RSS: 699 MB | Exec Mem: 3457 MB | Exec Samples: 0
# 100%|██████████| 1/1 [00:12<00:00, 12.51s/it]                                   
#    Повтор 6: 15.667 сек | Driver RSS: 699 MB | Exec Mem: 3457 MB | Exec Samples: 0
# 100%|██████████| 1/1 [00:12<00:00, 12.27s/it]                                   
#    Повтор 7: 15.447 сек | Driver RSS: 699 MB | Exec Mem: 4251 MB | Exec Samples: 0
# 100%|██████████| 1/1 [00:12<00:00, 12.15s/it]                                   
#    Повтор 8: 15.283 сек | Driver RSS: 699 MB | Exec Mem: 4317 MB | Exec Samples: 0
# 100%|██████████| 1/1 [00:12<00:00, 12.13s/it]                                   
#    Повтор 9: 15.286 сек | Driver RSS: 699 MB | Exec Mem: 4317 MB | Exec Samples: 0
# 100%|██████████| 1/1 [00:12<00:00, 12.35s/it]                                   
#    Повтор 10: 15.482 сек | Driver RSS: 699 MB | Exec Mem: 4982 MB | Exec Samples: 0

# 📊 n_rows = 750,000
# 100%|██████████| 1/1 [00:16<00:00, 16.32s/it]                                   
#    Повтор 1: 21.752 сек | Driver RSS: 1154 MB | Exec Mem: 4982 MB | Exec Samples: 0
# 100%|██████████| 1/1 [00:15<00:00, 15.89s/it]                                   
#    Повтор 2: 19.333 сек | Driver RSS: 1030 MB | Exec Mem: 4982 MB | Exec Samples: 0
# 100%|██████████| 1/1 [00:15<00:00, 15.31s/it]                                   
#    Повтор 3: 18.725 сек | Driver RSS: 869 MB | Exec Mem: 4982 MB | Exec Samples: 0
# 100%|██████████| 1/1 [00:15<00:00, 15.36s/it]                                   
#    Повтор 4: 18.749 сек | Driver RSS: 869 MB | Exec Mem: 4982 MB | Exec Samples: 0
# 100%|██████████| 1/1 [00:14<00:00, 14.99s/it]                                   
#    Повтор 5: 18.383 сек | Driver RSS: 724 MB | Exec Mem: 4982 MB | Exec Samples: 0
# 100%|██████████| 1/1 [00:15<00:00, 15.76s/it]                                   
#    Повтор 6: 19.163 сек | Driver RSS: 702 MB | Exec Mem: 4982 MB | Exec Samples: 0
# 100%|██████████| 1/1 [00:15<00:00, 15.07s/it]                                   
#    Повтор 7: 18.454 сек | Driver RSS: 702 MB | Exec Mem: 4982 MB | Exec Samples: 0
# 100%|██████████| 1/1 [00:15<00:00, 15.61s/it]                                   
#    Повтор 8: 19.059 сек | Driver RSS: 701 MB | Exec Mem: 4982 MB | Exec Samples: 0
# 100%|██████████| 1/1 [00:15<00:00, 15.38s/it]                                   
#    Повтор 9: 18.789 сек | Driver RSS: 701 MB | Exec Mem: 4982 MB | Exec Samples: 0
# 100%|██████████| 1/1 [00:15<00:00, 15.60s/it]                                   
#    Повтор 10: 19.004 сек | Driver RSS: 701 MB | Exec Mem: 4982 MB | Exec Samples: 0

# 📊 n_rows = 1,000,000
# 100%|██████████| 1/1 [00:17<00:00, 17.80s/it]                                   
#    Повтор 1: 24.071 сек | Driver RSS: 1382 MB | Exec Mem: 4982 MB | Exec Samples: 0
# 100%|██████████| 1/1 [00:17<00:00, 17.50s/it]                                   
#    Повтор 2: 21.159 сек | Driver RSS: 1179 MB | Exec Mem: 4982 MB | Exec Samples: 0
# 100%|██████████| 1/1 [00:17<00:00, 17.53s/it]                                   
#    Повтор 3: 21.180 сек | Driver RSS: 1105 MB | Exec Mem: 5432 MB | Exec Samples: 0
# 100%|██████████| 1/1 [00:17<00:00, 17.60s/it]                                   
#    Повтор 4: 21.240 сек | Driver RSS: 982 MB | Exec Mem: 5432 MB | Exec Samples: 0
# 100%|██████████| 1/1 [00:17<00:00, 17.86s/it]                                   
#    Повтор 5: 21.497 сек | Driver RSS: 982 MB | Exec Mem: 5432 MB | Exec Samples: 0
# 100%|██████████| 1/1 [00:17<00:00, 17.53s/it]                                   
#    Повтор 6: 21.141 сек | Driver RSS: 981 MB | Exec Mem: 5432 MB | Exec Samples: 0
# 100%|██████████| 1/1 [00:17<00:00, 17.98s/it]                                   
#    Повтор 7: 21.637 сек | Driver RSS: 981 MB | Exec Mem: 5432 MB | Exec Samples: 0
# 100%|██████████| 1/1 [00:17<00:00, 17.72s/it]                                   
#    Повтор 8: 21.360 сек | Driver RSS: 981 MB | Exec Mem: 5432 MB | Exec Samples: 0
# 100%|██████████| 1/1 [00:17<00:00, 17.36s/it]                                   
#    Повтор 9: 21.020 сек | Driver RSS: 981 MB | Exec Mem: 5432 MB | Exec Samples: 0
# 100%|██████████| 1/1 [00:16<00:00, 16.71s/it]                                   
#    Повтор 10: 20.333 сек | Driver RSS: 981 MB | Exec Mem: 5432 MB | Exec Samples: 0

# 📊 n_rows = 1,500,000
# 100%|██████████| 1/1 [00:20<00:00, 21.00s/it]                                   
#    Повтор 1: 28.762 сек | Driver RSS: 1947 MB | Exec Mem: 5432 MB | Exec Samples: 0
# 100%|██████████| 1/1 [00:21<00:00, 21.63s/it]                                   
#    Повтор 2: 25.740 сек | Driver RSS: 1597 MB | Exec Mem: 5432 MB | Exec Samples: 0
# 100%|██████████| 1/1 [00:21<00:00, 21.39s/it]                                   
#    Повтор 3: 25.502 сек | Driver RSS: 1509 MB | Exec Mem: 5432 MB | Exec Samples: 0
# 100%|██████████| 1/1 [00:21<00:00, 21.46s/it]                                   
#    Повтор 4: 25.631 сек | Driver RSS: 1620 MB | Exec Mem: 5432 MB | Exec Samples: 0
# 100%|██████████| 1/1 [00:21<00:00, 21.32s/it]                                   
#    Повтор 5: 25.447 сек | Driver RSS: 1582 MB | Exec Mem: 5432 MB | Exec Samples: 0
# 100%|██████████| 1/1 [00:20<00:00, 20.88s/it]                                   
#    Повтор 6: 24.994 сек | Driver RSS: 1543 MB | Exec Mem: 5432 MB | Exec Samples: 0
# 100%|██████████| 1/1 [00:21<00:00, 21.84s/it]                                   
#    Повтор 7: 25.974 сек | Driver RSS: 1612 MB | Exec Mem: 5432 MB | Exec Samples: 0
# 100%|██████████| 1/1 [00:21<00:00, 21.53s/it]                                   
#    Повтор 8: 25.669 сек | Driver RSS: 1577 MB | Exec Mem: 5583 MB | Exec Samples: 0
# 100%|██████████| 1/1 [00:20<00:00, 20.85s/it]                                   
#    Повтор 9: 25.004 сек | Driver RSS: 1646 MB | Exec Mem: 5583 MB | Exec Samples: 0
# 100%|██████████| 1/1 [00:20<00:00, 20.75s/it]                                   
#    Повтор 10: 24.842 сек | Driver RSS: 1586 MB | Exec Mem: 5583 MB | Exec Samples: 0

# 📊 n_rows = 2,000,000
# 100%|██████████| 1/1 [00:26<00:00, 26.76s/it]                                   
#    Повтор 1: 36.819 сек | Driver RSS: 2462 MB | Exec Mem: 5583 MB | Exec Samples: 0
# 100%|██████████| 1/1 [00:25<00:00, 25.93s/it]                                   
#    Повтор 2: 30.568 сек | Driver RSS: 1878 MB | Exec Mem: 5583 MB | Exec Samples: 0
# 100%|██████████| 1/1 [00:25<00:00, 25.72s/it]                                   
#    Повтор 3: 30.383 сек | Driver RSS: 1778 MB | Exec Mem: 5583 MB | Exec Samples: 0
# 100%|██████████| 1/1 [00:25<00:00, 25.95s/it]                                   
#    Повтор 4: 30.567 сек | Driver RSS: 1894 MB | Exec Mem: 5583 MB | Exec Samples: 0
# 100%|██████████| 1/1 [00:25<00:00, 25.88s/it]                                   
#    Повтор 5: 30.551 сек | Driver RSS: 1789 MB | Exec Mem: 5583 MB | Exec Samples: 0
# 100%|██████████| 1/1 [00:26<00:00, 26.71s/it]                                   
#    Повтор 6: 31.354 сек | Driver RSS: 1749 MB | Exec Mem: 5583 MB | Exec Samples: 0
# 100%|██████████| 1/1 [00:26<00:00, 26.79s/it]                                   
#    Повтор 7: 31.495 сек | Driver RSS: 733 MB | Exec Mem: 5583 MB | Exec Samples: 0
# 100%|██████████| 1/1 [00:25<00:00, 25.62s/it]                                   
#    Повтор 8: 30.239 сек | Driver RSS: 886 MB | Exec Mem: 5583 MB | Exec Samples: 0
# 100%|██████████| 1/1 [00:25<00:00, 25.93s/it]                                   
#    Повтор 9: 30.533 сек | Driver RSS: 886 MB | Exec Mem: 5583 MB | Exec Samples: 0
# 100%|██████████| 1/1 [00:25<00:00, 25.72s/it]                                   
#    Повтор 10: 30.358 сек | Driver RSS: 870 MB | Exec Mem: 6058 MB | Exec Samples: 0

# 📊 n_rows = 2,500,000
# 100%|██████████| 1/1 [00:29<00:00, 29.81s/it]                                   
#    Повтор 1: 41.243 сек | Driver RSS: 2923 MB | Exec Mem: 6058 MB | Exec Samples: 0
# 100%|██████████| 1/1 [00:30<00:00, 30.05s/it]                                   
#    Повтор 2: 35.193 сек | Driver RSS: 2193 MB | Exec Mem: 6058 MB | Exec Samples: 0
# 100%|██████████| 1/1 [00:29<00:00, 29.79s/it]                                   
#    Повтор 3: 34.939 сек | Driver RSS: 1737 MB | Exec Mem: 6058 MB | Exec Samples: 0
# 100%|██████████| 1/1 [00:30<00:00, 30.25s/it]                                   
#    Повтор 4: 35.319 сек | Driver RSS: 1329 MB | Exec Mem: 6289 MB | Exec Samples: 0
# 100%|██████████| 1/1 [00:30<00:00, 30.59s/it]                                   
#    Повтор 5: 35.713 сек | Driver RSS: 1326 MB | Exec Mem: 6778 MB | Exec Samples: 0
# 100%|██████████| 1/1 [00:30<00:00, 30.02s/it]                                   
#    Повтор 6: 35.144 сек | Driver RSS: 1326 MB | Exec Mem: 6903 MB | Exec Samples: 0
# 100%|██████████| 1/1 [00:30<00:00, 30.44s/it]                                   
#    Повтор 7: 35.595 сек | Driver RSS: 1449 MB | Exec Mem: 6903 MB | Exec Samples: 0
# 100%|██████████| 1/1 [00:30<00:00, 30.12s/it]                                   
#    Повтор 8: 36.210 сек | Driver RSS: 1448 MB | Exec Mem: 6903 MB | Exec Samples: 0
# 100%|██████████| 1/1 [00:30<00:00, 30.52s/it]                                   
#    Повтор 9: 35.573 сек | Driver RSS: 1448 MB | Exec Mem: 6903 MB | Exec Samples: 0
# 100%|██████████| 1/1 [00:31<00:00, 31.40s/it]                                   
#    Повтор 10: 36.565 сек | Driver RSS: 803 MB | Exec Mem: 6903 MB | Exec Samples: 0

# 📊 n_rows = 3,000,000
# 100%|██████████| 1/1 [00:36<00:00, 36.57s/it]                                   
#    Повтор 1: 50.504 сек | Driver RSS: 3080 MB | Exec Mem: 6903 MB | Exec Samples: 0
# 100%|██████████| 1/1 [00:35<00:00, 35.48s/it]                                   
#    Повтор 2: 41.160 сек | Driver RSS: 1143 MB | Exec Mem: 6903 MB | Exec Samples: 0
# 100%|██████████| 1/1 [00:35<00:00, 35.84s/it]                                   
#    Повтор 3: 41.539 сек | Driver RSS: 875 MB | Exec Mem: 6903 MB | Exec Samples: 0
# 100%|██████████| 1/1 [00:36<00:00, 36.37s/it]                                   
#    Повтор 4: 42.071 сек | Driver RSS: 989 MB | Exec Mem: 6903 MB | Exec Samples: 0
# 100%|██████████| 1/1 [00:35<00:00, 35.69s/it]                                   
#    Повтор 5: 41.412 сек | Driver RSS: 849 MB | Exec Mem: 6903 MB | Exec Samples: 0
# 100%|██████████| 1/1 [00:35<00:00, 35.60s/it]                                   
#    Повтор 6: 41.302 сек | Driver RSS: 1039 MB | Exec Mem: 6903 MB | Exec Samples: 0
# 100%|██████████| 1/1 [00:36<00:00, 36.27s/it]                                   
#    Повтор 7: 42.010 сек | Driver RSS: 951 MB | Exec Mem: 6903 MB | Exec Samples: 0
# 100%|██████████| 1/1 [00:36<00:00, 36.17s/it]                                   
#    Повтор 8: 41.860 сек | Driver RSS: 858 MB | Exec Mem: 6903 MB | Exec Samples: 0
# 100%|██████████| 1/1 [00:35<00:00, 35.47s/it]                                   
#    Повтор 9: 41.185 сек | Driver RSS: 976 MB | Exec Mem: 6903 MB | Exec Samples: 0
# 100%|██████████| 1/1 [00:34<00:00, 34.97s/it]                                   
#    Повтор 10: 40.647 сек | Driver RSS: 884 MB | Exec Mem: 6903 MB | Exec Samples: 0
                                                                                

# 📊 n_rows = 4,000,000
# 100%|██████████| 1/1 [00:46<00:00, 46.68s/it]                                   
#    Повтор 1: 64.348 сек | Driver RSS: 3914 MB | Exec Mem: 6903 MB | Exec Samples: 0
# 100%|██████████| 1/1 [00:46<00:00, 46.22s/it]                                   
#    Повтор 2: 53.010 сек | Driver RSS: 1039 MB | Exec Mem: 6903 MB | Exec Samples: 0
# 100%|██████████| 1/1 [00:44<00:00, 44.54s/it]                                   
#    Повтор 3: 51.389 сек | Driver RSS: 1062 MB | Exec Mem: 6903 MB | Exec Samples: 0
# 100%|██████████| 1/1 [00:43<00:00, 43.87s/it]                                   
#    Повтор 4: 50.546 сек | Driver RSS: 1113 MB | Exec Mem: 6960 MB | Exec Samples: 0
# 100%|██████████| 1/1 [00:43<00:00, 43.07s/it]                                   
#    Повтор 5: 49.736 сек | Driver RSS: 1188 MB | Exec Mem: 6960 MB | Exec Samples: 0
# 100%|██████████| 1/1 [00:43<00:00, 43.16s/it]                                   
#    Повтор 6: 49.757 сек | Driver RSS: 1231 MB | Exec Mem: 6960 MB | Exec Samples: 0
# 100%|██████████| 1/1 [00:43<00:00, 43.94s/it]                                   
#    Повтор 7: 50.600 сек | Driver RSS: 1084 MB | Exec Mem: 6960 MB | Exec Samples: 0
# 100%|██████████| 1/1 [00:43<00:00, 43.68s/it]                                   
#    Повтор 8: 50.317 сек | Driver RSS: 1140 MB | Exec Mem: 6960 MB | Exec Samples: 0
# 100%|██████████| 1/1 [00:45<00:00, 45.00s/it]                                   
#    Повтор 9: 51.590 сек | Driver RSS: 1077 MB | Exec Mem: 6960 MB | Exec Samples: 0
# 100%|██████████| 1/1 [00:45<00:00, 45.31s/it]                                   
#    Повтор 10: 52.659 сек | Driver RSS: 933 MB | Exec Mem: 6960 MB | Exec Samples: 0

# 🏁 Завершено: 16:32:45

In [ ]:
plot_comprehensive_results(results_df)
print_memory_analysis(results_df)

In [ ]:
results_df

In [ ]:
results_df.to_csv(f'{BenchmarkConfig.SAVE_PATH}/benchmark_data_2026_04_07_1737.csv', index=False)

In [ ]:
sp_s.stop()